## Question 1: Converting PDFs to Markdown

We want now to extract text from these books.

Install the `markitdown` library with PDF support:


In [3]:
# Install PDF conversion support
!uv pip install "markitdown[pdf]" --quiet

from pathlib import Path
from markitdown import MarkItDown

text_dir = Path("books_text")
text_dir.mkdir(exist_ok=True)

md = MarkItDown()
converted = []

for pdf_path in sorted(Path('.').glob('**/*.pdf')):
    output_path = text_dir / f"{pdf_path.stem}.md"
    result = md.convert(str(pdf_path))
    output_path.write_text(result.text_content, encoding="utf-8")
    converted.append((pdf_path, output_path))
    print(f"Converted: {pdf_path.name} -> {output_path.name}")

print(f"\nConverted {len(converted)} PDF(s) to Markdown files in {text_dir.resolve()}")

think_files = [
    path for path in text_dir.glob("*.md")
    if "think" in path.name.lower() and "python" in path.name.lower()
]

if think_files:
    target = sorted(think_files)[0]
    line_count = sum(1 for _ in target.open(encoding="utf-8"))
    print(f"Think Python markdown lines: {line_count}")
else:
    print("No Think Python markdown file found in books_text/")


2026-10-10 04:44:51.041546198 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+



Converted 0 PDF(s) to Markdown files in /workspaces/aiml_engineer_codebase/01-foundation/02-rag/books_text
No Think Python markdown file found in books_text/


Question 1. Converting PDFs to Markdown
We want now to extract text from these books.

Install the markitdown library with PDF support:

uv add 'markitdown[pdf]'
Copy
This library can convert various document formats including PDF to markdown or text format.

Convert all the downloaded PDFs to markdown files and save them to a books_text/ directory.

How many lines are in the extracted content from the "Think Python" book (2nd edition)?

In [4]:
from pathlib import Path
from markitdown import MarkItDown

# Find the project root, even if the notebook is running from a subdirectory
cwd = Path.cwd().resolve()
project_root = next(
    (path for path in (cwd, *cwd.parents) if (path / "books").is_dir()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not find a 'books' folder from {cwd}. "
        "Check that the notebook is running in the project workspace."
    )

pdfs = sorted((project_root / "books").glob("*.pdf"))
think_pdf = next(
    (pdf for pdf in pdfs if "think" in pdf.stem.lower() and "python" in pdf.stem.lower()),
    None,
)

if think_pdf is None:
    raise FileNotFoundError(
        f"No Think Python PDF found in {project_root / 'books'}. "
        f"PDFs found: {[pdf.name for pdf in pdfs]}"
    )

text_dir = project_root / "books_text"
text_dir.mkdir(exist_ok=True)

markdown_path = text_dir / f"{think_pdf.stem}.md"
markdown_path.write_text(
    MarkItDown().convert(str(think_pdf)).text_content,
    encoding="utf-8",
)

with markdown_path.open(encoding="utf-8") as file:
    line_count = sum(1 for _ in file)

print(f"Converted: {think_pdf}")
print(f"Markdown saved to: {markdown_path}")
print(f"Extracted lines: {line_count:,}")
print(f"Multiple-choice answer: {line_count:,}")

Converted: /workspaces/aiml_engineer_codebase/books/think_python_2e.pdf
Markdown saved to: /workspaces/aiml_engineer_codebase/books_text/think_python_2e.md
Extracted lines: 9,703
Multiple-choice answer: 9,703


Question 2. Chunking for RAG
For RAG we need to split documents into smaller chunks.

First, prepare your documents:

Read each markdown file from your books_text/ directory
Split the content into lines
Remove empty lines and lines that contain only whitespace
Turn each book into a dictionary with source (filename) and a content (list of non-empty lines)
After that, chunk it.

Use the gitsource package which provides the chunk_documents function. Install it if you don't have it:

uv add gitsource
Copy
The chunk_documents function uses a sliding window approach with these parameters:

size=100: number of items per chunk
step=50: how many items to move forward for each chunk
With size=100 and step=50, each chunk is about 4,400 characters or 780 words on average.

How many chunks are produced for the "Think Python" book (2nd edition) with these settings?

In [5]:
from pathlib import Path

from gitsource.chunking import chunk_documents

# Find the books_text directory from the notebook's current directory or its parents.
cwd = Path.cwd().resolve()
text_dir = next(
    (
        parent / "books_text"
        for parent in (cwd, *cwd.parents)
        if (parent / "books_text").is_dir()
        and any((parent / "books_text").glob("*.md"))
    ),
    None,
)
if text_dir is None:
    raise FileNotFoundError(
        f"Could not find a non-empty books_text directory from {cwd}. "
        "Run the PDF conversion step first."
    )

documents = []
for markdown_path in sorted(text_dir.glob("*.md")):
    lines = markdown_path.read_text(encoding="utf-8").splitlines()
    content = [line for line in lines if line.strip()]
    documents.append({"source": markdown_path.name, "content": content})

chunks = chunk_documents(documents, size=100, step=50)
think_python_chunks = [
    chunk
    for chunk in chunks
    if "think" in chunk["source"].lower()
    and "python" in chunk["source"].lower()
]
if not think_python_chunks:
    raise FileNotFoundError(
        f"No Think Python markdown file found in {text_dir}. "
        f"Markdown files found: {[path.name for path in text_dir.glob('*.md')]}"
    )

print(f"Prepared {len(documents)} book(s) from {text_dir}")
print(f"Think Python (2nd edition) chunks: {len(think_python_chunks):,}")

Prepared 1 book(s) from /workspaces/aiml_engineer_codebase/books_text
Think Python (2nd edition) chunks: 189


Question 3. Indexing with minsearch
Now we need to index our chunks so we can search through them.

We'll use minsearch. Install it if you don't have it:

uv add minsearch
Copy
Load all your chunked documents and create an index:

In [6]:
!uv add minsearch

Resolved 123 packages in 2ms
Checked 119 packages in 1ms


In [7]:
from minsearch import Index

def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)
    return documents

documents = prepare_documents(chunks)

index = Index(text_fields=["content"])
index.fit(documents)

print(f"Indexed documents (chunks): {len(documents):,}")

Indexed documents (chunks): 189


In [8]:
from minsearch import Index


def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)
    return documents


documents = prepare_documents(chunks)

index = Index(text_fields=["content"])
index.fit(documents)

print(f"Indexed documents (chunks): {len(documents):,}")

Indexed documents (chunks): 189


Question 4. Searching and RAG
Search our index.

results = index.search("python function definition", num_results=5)
Copy
Look at the top result. Which book did it come from?


Think Python 2e

Think DSP

Think Java

Physical Modeling in MATLAB

In [9]:
results = index.search("python function definition", num_results=5)

if not results:
    raise ValueError("The index returned no results for the search query.")

top_result = results[0]
source = top_result["source"]

book_titles = {
    "think_python_2e.md": "Think Python 2e",
    "think_dsp.md": "Think DSP",
    "think_java_2e.md": "Think Java",
    "physical_modeling_in_matlab.md": "Physical Modeling in MATLAB",
}

print(f"Top result book: {book_titles.get(source, source)}")
print(f"Source file: {source}")
print(f"Top result excerpt: {top_result['content'][:500]}")

Top result book: Think Python 2e
Source file: think_python_2e.md
Top result excerpt: returningitdirectly:
| def is_divisible(x, | y):      |     |
| ------------------- | -------- | --- |
| return x            | % y == 0 |     |
Booleanfunctionsareoftenusedinconditionalstatements:
| if is_divisible(x, | y):          |        |
| ------------------ | ------------ | ------ |
| print('x           | is divisible | by y') |
Itmightbetemptingtowritesomethinglike:
| if is_divisible(x, | y) ==        | True:  |
| ------------------ | ------------ | ------ |
| print('x           | is div


Question 5. Full RAG
We're ready to do RAG.

This is the code we wrote in the lessons (or similar to it - but please use this code to make sure the results are reproducible):

import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer
Copy
Do RAG for "python function definition". What's the response?

You don't have to use OpenAI, you can use an alternative provider.

Next, change these functions to also return the number of input and output tokens.

How many input tokens did we use for this one RAG query?


5944

7944

9944

11944

In [10]:
import json

from openai import OpenAI

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(question=question, context=context).strip()


def search(question):
    return index.search(question, num_results=5)


openai_client = OpenAI()


def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]
    response = openai_client.responses.create(model=model, input=messages)
    if response.usage is None:
        raise RuntimeError("The model response did not include token usage.")
    return (
        response.output_text,
        response.usage.input_tokens,
        response.usage.output_tokens,
    )


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)


query = "python function definition"
answer, input_tokens, output_tokens = rag(query)

print(f"Question: {query}")
print(f"Answer: {answer}")
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")

Question: python function definition
Answer: In Python, a function is defined using the `def` keyword, followed by the function name and parentheses which may include parameters. The function's body is indented beneath the definition line and contains the statements that execute when the function is called.

Here’s an example of a simple function definition:

```python
def is_divisible(x, y):
    return x % y == 0
```

In this case, the function `is_divisible` takes two parameters `x` and `y`, and returns `True` if `x` is divisible by `y` and `False` otherwise. 

To use a function, you call it by name with arguments:

```python
if is_divisible(6, 3):
    print('6 is divisible by 3')
```

You can also define your own functions like `is_between`, which can check if a number falls within a specified range:

```python
def is_between(x, y, z):
    return x <= y <= z
```

This function would return `True` if `y` is between `x` and `z`, inclusive.
Input tokens: 7585
Output tokens: 231


Question 6. Structured vs Unstructured Output
We now use structured outputs with a Pydantic model.

Define the response model:

from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")
Copy
Modify the llm and rag functions from Question 5 to use structured outputs.

do RAG for "python function definition"
look at the number of input tokens
compare the number with the results from Q5
How many MORE input tokens does the structured output version use compared to the unstructured version?


86

186

286

386

In [12]:
from typing import Literal

from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(
        description="The category of the answer"
    )
    followup_questions: list[str] = Field(description="Suggested follow-up questions")


def llm_structured(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]
    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse,
    )
    if response.usage is None:
        raise RuntimeError("The model response did not include token usage.")
    if response.output_parsed is None:
        raise RuntimeError("The model response did not contain parsed structured output.")
    return (
        response.output_parsed,
        response.usage.input_tokens,
        response.usage.output_tokens,
    )


def rag_structured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(prompt, instructions)


query = "python function definition"
unstructured_input_tokens = input_tokens
structured_answer, structured_input_tokens, structured_output_tokens = rag_structured(query)
additional_input_tokens = structured_input_tokens - unstructured_input_tokens

print(f"Question: {query}")
print(f"Structured answer:\n{structured_answer.model_dump_json(indent=2)}")
print(f"Unstructured input tokens (Question 5): {unstructured_input_tokens}")
print(f"Structured input tokens: {structured_input_tokens}")
print(f"Additional input tokens: {additional_input_tokens}")

Question: python function definition
Structured answer:
{
  "answer": "In Python, a function is defined using the `def` keyword followed by the function name and parentheses which may include parameters. The basic structure for defining a function looks like this:\n\n```python\n\ndef function_name(parameters):\n    \"\"\"Docstring explaining the function.\"\"\"\n    # Function body, where the logic is implemented\n    return value  # Optional\n```  \n\nFor example, here’s a simple function that checks if a number is divisible by another:\n\n```python\n\ndef is_divisible(x, y):\n    return x % y == 0\n```  \n\nYou can call this function with two arguments, and it will return `True` if `x` is divisible by `y`, and `False` otherwise:\n\n```python\nresult = is_divisible(6, 3)  # This will return True\n```",
  "found_answer": true,
  "confidence": 0.9,
  "confidence_explanation": "The context provides information about function definitions in Python, illustrating the structure, examples, an